In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import pandas as pd
import numpy as np
import re

# 1. Data Loading
url = "https://www.iposcoop.com/ipos-recently-filed/"
tables = pd.read_html(url)
df = next(t for t in tables if 'Expected To Trade' in t.columns or 'Expected to Trade' in t.columns)
df.columns = df.columns.str.strip()

# Filter withdrawn
withdrawn_df = df[df['Expected To Trade'].str.strip() == 'Withdrawn'].copy()

# 2. Company Classification (Order matters!)
def classify_company(name):
    name = str(name)
    if "Technologies" in name: return "Technologies"
    elif "Acquisition Corp" in name or "Acquisition Corporation" in name or "Corp" in name: return "Acquisition Corp"
    elif "Inc" in name or "Incorporated" in name: return "Inc."
    elif "Group" in name: return "Group"
    elif "Ltd" in name or "Limited" in name: return "Limited"
    elif "Holdings" in name or "Holding" in name: return "Holdings"
    else: return "Other"

withdrawn_df['Company Type'] = withdrawn_df['Company'].apply(classify_company)

# 3. Price Parsing
def parse_price(val):
    if pd.isna(val) or str(val).strip() in ['-', '']: return np.nan
    match = re.search(r'[\d\.]+', str(val))
    return float(match.group()) if match else np.nan

withdrawn_df['Price Low Num'] = withdrawn_df['Price Low'].apply(parse_price)
withdrawn_df['Price High Num'] = withdrawn_df['Price High'].apply(parse_price)
withdrawn_df['Avg_price'] = (withdrawn_df['Price Low Num'] + withdrawn_df['Price High Num']) / 2

# 4. Numeric Conversion
def clean_numeric(val):
    if pd.isna(val) or str(val).strip() in ['-', '']: return np.nan
    return float(re.sub(r'[\$,]', '', str(val)))

withdrawn_df['Shares Num'] = withdrawn_df['Shares (millions)'].apply(clean_numeric)
withdrawn_df['Est Vol Num'] = withdrawn_df['Est $ Vol (millions)'].apply(clean_numeric)

# 5. Value Calculation
withdrawn_df['Shares_offered_value'] = np.where(
    withdrawn_df['Shares Num'].notna() & withdrawn_df['Avg_price'].notna(),
    withdrawn_df['Shares Num'] * withdrawn_df['Avg_price'],
    withdrawn_df['Est Vol Num']
)

# 6. Aggregation
result = withdrawn_df.groupby('Company Type')['Shares_offered_value'].sum().reset_index()
result = result.sort_values(by='Shares_offered_value', ascending=False)
print(result)

       Company Type  Shares_offered_value
0  Acquisition Corp              499.9850
3              Inc.              351.0000
2          Holdings              311.6575
5             Other              290.4450
4           Limited              219.2500
6      Technologies              184.9000
1             Group               49.3750


Answer is Acquisition Corp with the value of 499.985 million


In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

# 1. Data Loading
url = "https://www.iposcoop.com/2025-pricings/"
tables = pd.read_html(url)
df = next(t for t in tables if 'Offer Date' in t.columns or 'Offer date' in t.columns)

# 2. Filtering
df['Offer Date'] = pd.to_datetime(df['Offer Date'])
df['Return'] = df['Return'].astype(str).str.replace('%', '').astype(float)
filtered_df = df[(df['Offer Date'] < '2025-09-01') & (df['Return'] != 0.0)].copy()
tickers = filtered_df['Symbol'].unique().tolist()

# 3. Data Download (with error handling for delisted stocks)
all_data = []
for ticker in tickers:
    try:
        # Download data; auto_adjust=False to keep raw Close prices as per standard financial calc
        data = yf.download(ticker, start='2025-01-01', end='2026-09-15', progress=False, auto_adjust=False)
        if not data.empty:
            data['Ticker'] = ticker
            all_data.append(data)
    except Exception:
        pass # Skip delisted/missing tickers

stocks_df = pd.concat(all_data, axis=0).reset_index()
stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])

# 4. Feature Engineering (Grouped by Ticker to prevent cross-contamination)
stocks_df['growth_252d'] = stocks_df.groupby('Ticker')['Close'].transform(lambda x: x / x.shift(252))
stocks_df['volatility'] = stocks_df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(30).std() * np.sqrt(252))

# 5. Sharpe Ratio Calculation
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

# 6. Final Analysis
target_date = pd.to_datetime('2026-09-11').date()
final_df = stocks_df[stocks_df['Date'].dt.date == target_date]

print("Descriptive Statistics for Sharpe Ratio:")
print(final_df['Sharpe'].describe())
print(f"Median Sharpe Ratio: {final_df['Sharpe'].median():.4f}")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}

1 Failed download:
['MJID']: YFTzMissingError('possibly delisted; no timezone found')
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}

1 Failed download:
['EMPG']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['CAEP']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['PTNM']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['AHL']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['CEPT']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['SDM']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['TBH']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['WGRX']: YFPricesMissingError('p

ValueError: Cannot set a DataFrame with multiple columns to the single column growth_252d

In [3]:
import pandas as pd
import numpy as np
import yfinance as yf

# 1. Data Loading
url = "https://www.iposcoop.com/2025-pricings/"
tables = pd.read_html(url)
df = next(t for t in tables if 'Offer Date' in t.columns or 'Offer date' in t.columns)

# 2. Filtering
df['Offer Date'] = pd.to_datetime(df['Offer Date'])
df['Return'] = df['Return'].astype(str).str.replace('%', '').astype(float)
filtered_df = df[(df['Offer Date'] < '2025-09-01') & (df['Return'] != 0.0)].copy()
tickers = filtered_df['Symbol'].unique().tolist()

print(f"Downloading {len(tickers)} tickers in chunks to avoid timeouts...")

# 3. Data Download (in chunks to prevent HTTP 404/Timeouts)
all_data = []
chunk_size = 20
for i in range(0, len(tickers), chunk_size):
    chunk = tickers[i:i+chunk_size]
    try:
        # auto_adjust=False ensures we get the raw Close price for accurate calculations
        data = yf.download(chunk, start='2025-01-01', end='2026-09-15', progress=False, auto_adjust=False)
        if not data.empty:
            all_data.append(data)
    except Exception as e:
        print(f"Skipped chunk {i} due to error: {e}")

if not all_data:
    raise ValueError("No data was downloaded. Check your internet connection or ticker list.")

# Concatenate all chunks
stocks_df = pd.concat(all_data, axis=0)

# --- CRITICAL FIX: Handle MultiIndex Columns ---
if isinstance(stocks_df.columns, pd.MultiIndex):
    # Stack the ticker level (level 1) to rows to create a clean long-format DataFrame
    stocks_df = stocks_df.stack(level=1).reset_index()
    # Rename the generated ticker column (usually 'level_1' or 'Ticker')
    ticker_col = 'Ticker' if 'Ticker' in stocks_df.columns else 'level_1'
    stocks_df.rename(columns={ticker_col: 'Ticker'}, inplace=True)
else:
    # Fallback for single-ticker downloads
    stocks_df = stocks_df.reset_index()
    stocks_df['Ticker'] = tickers[0]

# Ensure Date is a datetime object and clean missing data
stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])
stocks_df = stocks_df.dropna(subset=['Close'])

# Sort by Ticker and Date to ensure shift/rolling calculations are accurate
stocks_df = stocks_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# 4. Feature Engineering (Using named functions to prevent lambda DataFrame bugs)
def calc_growth(x):
    return x / x.shift(252)

def calc_volatility(x):
    return x.rolling(30).std() * np.sqrt(252)

stocks_df['growth_252d'] = stocks_df.groupby('Ticker')['Close'].transform(calc_growth)
stocks_df['volatility'] = stocks_df.groupby('Ticker')['Close'].transform(calc_volatility)

# 5. Sharpe Ratio Calculation
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

# 6. Final Analysis
target_date = pd.to_datetime('2026-09-11').date()
final_df = stocks_df[stocks_df['Date'].dt.date == target_date]

print("\n--- Descriptive Statistics for Sharpe Ratio ---")
print(final_df['Sharpe'].describe())
print(f"\nMedian Sharpe Ratio: {final_df['Sharpe'].median():.4f}")


1 Failed download:
['MJID']: YFTzMissingError('possibly delisted; no timezone found')

2 Failed downloads:
['CAEP', 'EMPG']: YFTzMissingError('possibly delisted; no timezone found')

4 Failed downloads:
['CEPT', 'AHL', 'SDM', 'PTNM']: YFTzMissingError('possibly delisted; no timezone found')

3 Failed downloads:
['AGH', 'TBH']: YFTzMissingError('possibly delisted; no timezone found')
['WGRX']: YFPricesMissingError('possibly delisted; no price data found  (1d 2025-01-01 -> 2026-09-15)')

4 Failed downloads:
['SKBL', 'MTSR', 'MCTR', 'EPWK']: YFTzMissingError('possibly delisted; no timezone found')
/tmp/ipykernel_58/1001370278.py:40: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stocks_df = stocks_df.stack(level=1).reset_index()



--- Descriptive Statistics for Sharpe Ratio ---
count    130.000000
mean            inf
std             NaN
min       -0.040147
25%        0.012042
50%        0.049016
75%        0.133619
max             inf
Name: Sharpe, dtype: float64

Median Sharpe Ratio: 0.0490


In [7]:
import pandas as pd
import numpy as np

# --- SANITY CHECK ---
# Ensure the DataFrame from Question 2 is properly formatted
if 'Ticker' not in stocks_df.columns or 'Close' not in stocks_df.columns:
    raise ValueError("stocks_df is missing 'Ticker' or 'Close' columns. Please re-run the fully corrected Question 2 code first.")

# Sort to ensure strict chronological order within each ticker
stocks_df = stocks_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# 1. Identify Entry Points: Get the very first closing price for each ticker
# Using .map() is much safer than .merge() and avoids pandas index-alignment bugs
first_close_map = stocks_df.groupby('Ticker')['Close'].first().to_dict()
stocks_df['First_Close'] = stocks_df['Ticker'].map(first_close_map)  # <-- Fixed missing parenthesis here

# 2. Feature Engineering: Calculate 12 future growth columns
for k in range(1, 13):
    days = 21 * k
    # Shift negative to look into the future from the current row
    future_close = stocks_df.groupby('Ticker')['Close'].shift(-days)
    stocks_df[f'future_growth_{k}_m'] = future_close / stocks_df['First_Close']

# 3. Data Alignment: Filter to keep ONLY the min_date (IPO date) records for each ticker
min_dates = stocks_df.groupby('Ticker')['Date'].min().reset_index()

# Inner join ensures we only evaluate the growth starting exactly from the first trading day
iso_df = stocks_df.merge(min_dates, on=['Ticker', 'Date'], how='inner')

# 4. Median Analysis
growth_cols = [f'future_growth_{k}_m' for k in range(1, 13)]
medians = iso_df[growth_cols].median()

optimal_month = medians.idxmax()
max_median_growth = medians.max()

print("\n--- Fixed Months Holding Strategy Results ---")
print(f"Optimal holding period: {optimal_month}")
print(f"Maximum median growth value: {max_median_growth:.4f}")

print("\nMedian growth by month:")
print(medians.round(4))


--- Fixed Months Holding Strategy Results ---
Optimal holding period: future_growth_1_m
Maximum median growth value: 0.9354

Median growth by month:
future_growth_1_m     0.9354
future_growth_2_m     0.8930
future_growth_3_m     0.8272
future_growth_4_m     0.7304
future_growth_5_m     0.6909
future_growth_6_m     0.7009
future_growth_7_m     0.6600
future_growth_8_m     0.6035
future_growth_9_m     0.5803
future_growth_10_m    0.5333
future_growth_11_m    0.4818
future_growth_12_m    0.4918
dtype: float64
